# Figures and supplementary workbook

Draws every figure of *Diverging Pre-Pandemic Mortality Trends: Age-Specific and
Cause-Specific Patterns Across High-Income Countries* and of its Supplementary
File 1, and writes the workbook of Supplementary File 2. Everything is read from
the estimates tables in `estimates/`, written by `mortality_estimates.py`; no
model is fitted here.

| output | in the paper |
|---|---|
| `figures/Figure1_AllCause_APC_heatmap.png` (and `.tif`) | Figure 1 |
| `figures/Figure2_Chapter_APC_heatmap.png` (and `.tif`) | Figure 2 |
| `figures/Figure3_TrendCategory_shares.png` (and `.tif`) | Figure 3 |
| `figures/FigureS1_Linearity.png` | Supplementary File 1, Figure S1 |
| `figures/FigureS2_Estimator_comparison.png` | Figure S2 |
| `figures/FigureS3_Level_relative_to_median.png` | Figure S3 |
| `figures/FigureS4_AllCause_forest_all_ages.png` | Figure S4 |
| `figures/FigureS5_Detectability.png` | Figure S5 |
| `figures/FigureS6_Precision_funnel.png` | Figure S6 |
| `figures/FigureS7_Sex_APC_heatmaps.png` | Figure S7 |
| `figures/FigureS8_Sex_dumbbell.png` | Figure S8 |
| `figures/FigureS9_Sex_resolvability.png` | Figure S9 |
| `figures/FigureS10_Sex_chapter_differences.png` | Figure S10 |
| `workbooks/Supplementary_estimates.xlsx` | Supplementary File 2 |

Run the cells in order from the repository root.

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.legend_handler import HandlerPatch
from mpl_toolkits.axes_grid1 import make_axes_locatable
from PIL import Image

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------- locations
DATA_DIR = Path("estimates")             # tables written by mortality_estimates.py
REF_DIR = Path("reference")              # published breakpoints (Figure S1 section)
FIG_DIR = Path("figures")
BOOK_DIR = Path("workbooks")
FIG_DIR.mkdir(exist_ok=True)
BOOK_DIR.mkdir(exist_ok=True)
DPI = 300

EST_ALL_CAUSE = DATA_DIR / "estimates_all_cause.csv"
EST_CHAPTER = DATA_DIR / "estimates_chapter.csv"
EST_CODE = DATA_DIR / "estimates_code.csv.gz"
EST_SEX_ALL_CAUSE = DATA_DIR / "estimates_sex_all_cause.csv"
EST_SEX_CHAPTER = DATA_DIR / "estimates_sex_chapter.csv"
BREAKPOINT_FILE = REF_DIR / "McCartney_breakpoints_our_countries.csv"

# ---------------------------------------------------------------- analysis
# Materiality threshold, %/year. Must match the value used when the tables
# were built (mortality_estimates.py --apc-threshold).
APC_THRESHOLD = 1.0
ESTIMABLE = ("ok", "sparse")
AGE_ORDER = ["0", "1-14", "15-29", "30-44", "45-59", "60-74", "75+"]

# Figure 2: a chapter gets a row in an age group if it accounts for at least
# this share of deaths in at least one country and rises materially somewhere.
MIN_CHAPTER_SHARE_PCT = 1.0
# Figure 3: an age group is drawn if, on average over countries, at least this
# share of deaths falls in a resolved class.
FIG3_MIN_RESOLVED_PCT = 2.0
FIG3_PANEL_HEIGHT = 2.15                 # inches per panel

# ---------------------------------------------------------------- house style
COL_INC = "#ff6b6b"      # increasing
COL_NEU = "#9aa0a6"      # stagnation / no material change
COL_DEC = "#4dabf7"      # decreasing
COL_INCONCL = "#dee2e6"  # inconclusive
COL_NA = "#f8f9fa"       # not estimable (drawn with diagonal hatching)
COL_MARK = "#212529"     # dots and open circles

CMAP_APC = LinearSegmentedColormap.from_list(
    "apc", ["#08519c", COL_DEC, "#cfe8fb", "#f7f7f7", "#ffd4d4", COL_INC, "#a01010"])
CMAP_APC.set_bad(COL_NA)

FS_TICK = 16
FS_LABEL = 18
FS_PANEL = 18
FS_AXIS = 20

mpl.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": DPI,
    "savefig.bbox": "tight",
    "font.size": 13,
    "axes.titlesize": FS_PANEL,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Category colours for the forest plot (Figure S4). "significant but small"
# is coloured by the sign of the estimate.
COL_SMALL_UP = "#ffc9c9"
COL_SMALL_DOWN = "#c5e2fb"
CATEGORY_COLOURS = {
    "increasing": COL_INC,
    "significant but small (increase)": COL_SMALL_UP,
    "significant but small (decrease)": COL_SMALL_DOWN,
    "inconclusive": COL_INCONCL,
    "no material change": COL_NEU,
    "decreasing": COL_DEC,
    "not estimable": COL_NA,
}


def signed_category(cat, apc):
    """Split 'significant but small' by the sign of the estimate, for display."""
    if cat == "significant but small" and np.isfinite(apc):
        return f"significant but small ({'increase' if apc > 0 else 'decrease'})"
    return cat


def category_colour(cat, apc=np.nan):
    return CATEGORY_COLOURS.get(signed_category(cat, apc), COL_NEU)


print("matplotlib", mpl.__version__, "| seaborn", sns.__version__,
      "| pandas", pd.__version__)

## Loading

The analysis countries are read from the chapter table: those with at least one
estimable cause-specific cell. `fdr_sig` marks a material change, one that
survives Benjamini–Hochberg control and is at least `APC_THRESHOLD` in magnitude.

In [ ]:
def analysis_countries(path=EST_CHAPTER):
    d = pd.read_csv(path, usecols=["country", "status"], low_memory=False)
    return sorted(d.loc[d.status.isin(ESTIMABLE), "country"].unique())


ANALYSIS_COUNTRIES = analysis_countries()


def load_estimates(path, usecols=None):
    df = pd.read_csv(path, usecols=usecols, low_memory=False)
    df = df[df.country.isin(ANALYSIS_COUNTRIES)].copy()
    df["age_group"] = pd.Categorical(df.age_group, categories=AGE_ORDER, ordered=True)
    df = df[df.age_group.notna()]
    df["estimable"] = df.status.isin(ESTIMABLE)
    df["fdr_sig"] = df.p_bh.lt(0.05) & df.apc.abs().ge(APC_THRESHOLD) & df.estimable
    print(f"{Path(path).name}: {len(df)} rows, {df.country.nunique()} countries")
    return df


def pivot(df, values, index="age_group", columns="country", countries=None):
    m = df.pivot_table(index=index, columns=columns, values=values,
                       aggfunc="first", observed=False)
    if countries is None:
        countries = sorted(m.columns)
    return m.reindex(index=[a for a in AGE_ORDER if a in m.index], columns=countries)


CODE_COLS = ["country", "age_group", "cause", "cause_label", "total_deaths",
             "apc", "apc_lo", "apc_hi", "p", "p_bh", "mde80", "se",
             "share_of_group_deaths_pct", "status", "category"]

all_cause = load_estimates(EST_ALL_CAUSE)
chapters = load_estimates(EST_CHAPTER)
codes = load_estimates(EST_CODE, usecols=CODE_COLS)
COUNTRIES = sorted(all_cause.country.unique())
print(f"{len(COUNTRIES)} countries")

## Drawing helpers

`draw_apc_heatmap` draws the colour map (annual percentage change, diagonal
hatching where no estimate is possible); `draw_class_marks` adds the marks of
Figures 1 and 2: a filled dot for a material change, an open circle for a change
that survives false discovery rate control but is below the threshold, and
cross-hatching where no change was detected and the interval lies within the
threshold. An inconclusive cell carries no mark.

In [ ]:
def symmetric_vmax(values, q=0.98, minimum=1.0, step=0.5):
    """Rounded symmetric colour limit from a quantile of |values|."""
    v = pd.Series(values).abs().replace([np.inf, -np.inf], np.nan).dropna()
    if v.empty:
        return minimum
    raw = float(np.nanquantile(v, q))
    return max(minimum, float(np.ceil(raw / step) * step))


def draw_apc_heatmap(ax, mat_apc, mat_sig=None, mat_ok=None, vmax=None,
                     cbar=False, cbar_ax=None, xlabels=True, ylabels=True,
                     fs_tick=FS_TICK, dot_size=26, hatch="////"):
    """Colour map of annual percentage change, with not-estimable cells hatched
    and, if mat_sig is given, a dot on each material change."""
    vmax = symmetric_vmax(mat_apc.values.ravel()) if vmax is None else vmax
    plotted = mat_apc.astype(float).copy()
    if mat_ok is not None:
        plotted = plotted.where(mat_ok.fillna(False).astype(bool))
    plotted = plotted.clip(lower=-vmax, upper=vmax)

    ax.set_facecolor(COL_NA)
    sns.heatmap(plotted, ax=ax, cmap=CMAP_APC, vmin=-vmax, vmax=vmax, center=0,
                cbar=cbar, cbar_ax=cbar_ax,
                cbar_kws={"label": "Annual change in mortality rate (%/year)"} if cbar else None,
                linewidths=0.4, linecolor="white", square=False)

    if mat_ok is not None:
        bad = ~mat_ok.fillna(False).astype(bool).values
        for i, j in zip(*np.where(bad)):
            ax.add_patch(mpatches.Rectangle((j, i), 1, 1, fill=True, facecolor=COL_NA,
                                            hatch=hatch, edgecolor="#adb5bd",
                                            linewidth=0.0, zorder=3))
    if mat_sig is not None:
        sig = mat_sig.fillna(False).astype(bool).values
        ys, xs = np.where(sig)
        ax.scatter(xs + 0.5, ys + 0.5, s=dot_size, c=COL_MARK, marker="o",
                   zorder=4, linewidths=0)

    ax.set_xlabel("")
    ax.set_ylabel("")
    if xlabels:
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="center",
                           va="top", fontsize=fs_tick)
    else:
        ax.set_xticklabels([])
        ax.tick_params(axis="x", length=0)
    if ylabels:
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=fs_tick)
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis="y", length=0)
    return vmax


def draw_class_marks(ax, mat_cat, mat_ok, circle_size=34):
    """Open circle on 'significant but small', cross-hatching on 'no material
    change'. Dots for material changes are drawn by draw_apc_heatmap."""
    ok = mat_ok.fillna(False).astype(bool).values
    cat = mat_cat.values
    small = (cat == "significant but small") & ok
    nomat = (cat == "no material change") & ok
    ys, xs = np.where(small)
    ax.scatter(xs + 0.5, ys + 0.5, s=circle_size, facecolors="none",
               edgecolors=COL_MARK, linewidths=1.1, zorder=4)
    for i, j in zip(*np.where(nomat)):
        ax.add_patch(mpatches.Rectangle((j, i), 1, 1, fill=False, hatch="xx",
                                        edgecolor="#495057", linewidth=0.0,
                                        zorder=3.5))
    return int(small.sum()), int(nomat.sum())


def class_legend_handles():
    return [
        plt.Line2D([], [], marker="o", linestyle="none", markersize=7,
                   markerfacecolor=COL_MARK, markeredgecolor="none",
                   label=f"Material change ($\\geq${APC_THRESHOLD:g}%/year, FDR-significant)"),
        plt.Line2D([], [], marker="o", linestyle="none", markersize=7,
                   markerfacecolor="none", markeredgecolor=COL_MARK, markeredgewidth=1.1,
                   label=f"FDR-significant, below {APC_THRESHOLD:g}%/year"),
        mpatches.Patch(facecolor="white", hatch="xxx", edgecolor="#343a40",
                       label=f"No change detected, interval within $\\pm${APC_THRESHOLD:g}%/year"),
        mpatches.Patch(facecolor=COL_NA, hatch="////", edgecolor="#adb5bd",
                       label="Not estimable (too few deaths)"),
    ]


def dot_legend_handles():
    """Legend for maps that carry the dot only (Figures S7 and S10)."""
    return [
        plt.Line2D([], [], marker="o", color="none", markerfacecolor=COL_MARK,
                   markersize=7,
                   label=f"FDR-significant and |change| $\\geq$ {APC_THRESHOLD:g} %/year"),
        mpatches.Patch(facecolor=COL_NA, hatch="////", edgecolor="#adb5bd",
                       label="Not estimable (too few deaths)"),
    ]


def panel_label(ax, text, fs=FS_PANEL):
    ax.text(0.004, 1.01, text, transform=ax.transAxes, va="bottom", ha="left",
            fontsize=fs, fontweight="bold")


def savefig(fig, name):
    """Write a PNG at DPI; main-text figures (Figure1..3) are also written as an
    RGB, LZW-compressed TIFF converted from that PNG, so the two are one image."""
    path = FIG_DIR / name
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    print("wrote", path)
    if re.match(r"Figure\d", name) and path.suffix.lower() == ".png":
        tif = path.with_suffix(".tif")
        Image.open(path).convert("RGB").save(tif, dpi=(DPI, DPI), compression="tiff_lzw")
        print("wrote", tif)


def panel_ages(df, order=AGE_ORDER):
    have = set(df.age_group.dropna().unique())
    return [a for a in order if a in have]


def panel_letter(i):
    return "abcdefghijkl"[i]

## Figure 1 — all-cause trends by age group and country

In [ ]:
mat_apc = pivot(all_cause, "apc", countries=COUNTRIES)
mat_sig = pivot(all_cause, "fdr_sig", countries=COUNTRIES)
mat_ok = pivot(all_cause, "estimable", countries=COUNTRIES)
mat_cat = pivot(all_cause, "category", countries=COUNTRIES)
vmax = symmetric_vmax(all_cause.apc, q=0.95)

mpl.rcParams["hatch.linewidth"] = 0.7
fig, ax = plt.subplots(figsize=(max(12, 0.42 * len(COUNTRIES)), 5.6))
cax = make_axes_locatable(ax).append_axes("right", size="1.1%", pad=0.10)
draw_apc_heatmap(ax, mat_apc, mat_sig, mat_ok, vmax=vmax, cbar=True, cbar_ax=cax)
n_small, n_nomat = draw_class_marks(ax, mat_cat, mat_ok)

ax.invert_yaxis()                       # youngest age group at the bottom
ax.set_ylabel("Age group", fontsize=FS_LABEL)
cax.set_ylabel("Annual change in mortality rate (%/year)", fontsize=12, labelpad=6)
cax.tick_params(labelsize=11)
ax.legend(handles=class_legend_handles(), loc="lower center",
          bbox_to_anchor=(0.5, 1.04), ncol=2, fontsize=12, frameon=True)
savefig(fig, "Figure1_AllCause_APC_heatmap.png")
plt.show()

okv = mat_ok.fillna(False).astype(bool).values
n_dot = int((mat_sig.fillna(False).astype(bool).values & okv).sum())
n_est = int(okv.sum())
print(f"{n_est} estimable cells: {n_dot} filled dot, {n_small} open circle, "
      f"{n_nomat} cross-hatched, {n_est - n_dot - n_small - n_nomat} unmarked")

## Figure 2 — cause-specific trends by ICD-10 chapter

Within each age group a chapter is shown where at least one country has a
material increase and the chapter accounts for at least `MIN_CHAPTER_SHARE_PCT`
of deaths in at least one country. Marks as in Figure 1.

In [ ]:
CHAPTER_ROMAN = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X",
                 "XI", "XII", "XIII", "XIV", "XV", "XVI", "XVII", "XVIII",
                 "XIX", "XX", "XXI", "XXII"]
label_of = chapters.drop_duplicates("cause").set_index("cause").cause_label


def nice_chapter(c):
    return f"{c}. {label_of.get(c, '')}".rstrip(". ")


def chapter_share_range(df, age_group, causes):
    d = df[df.age_group == age_group]
    g = d.groupby("cause", observed=True).share_of_group_deaths_pct.agg(["min", "max"])
    return g.reindex(causes)


def chapter_matrices(df, age_group, countries, min_share_pct=MIN_CHAPTER_SHARE_PCT):
    d = df[df.age_group == age_group]
    share = d.groupby("cause", observed=True).share_of_group_deaths_pct.max()
    d = d[d.cause.isin(share[share >= min_share_pct].index)]
    mats = {v: d.pivot_table(index="cause", columns="country", values=v,
                             aggfunc="first", observed=False)
            for v in ("apc", "fdr_sig", "estimable", "category")}
    rows = [c for c in CHAPTER_ROMAN if c in mats["apc"].index]
    cols = [c for c in countries if c in mats["apc"].columns]
    mats = {k: m.reindex(rows, columns=cols) for k, m in mats.items()}
    rising = ((mats["apc"] > 0) & mats["fdr_sig"].fillna(False).astype(bool)).any(axis=1)
    keep = rising[rising].index.tolist()
    mats = {k: m.loc[keep].rename(index=nice_chapter) for k, m in mats.items()}
    return mats["apc"], mats["fdr_sig"], mats["estimable"], mats["category"]


mats = {age: chapter_matrices(chapters, age, COUNTRIES) for age in panel_ages(chapters)}
dropped = [a for a, m in mats.items() if m[0].empty]
mats = {a: m for a, m in mats.items() if not m[0].empty}
print("age groups with no chapter meeting the rule:", dropped)
vmax_ch = symmetric_vmax(np.concatenate([m[0].values.ravel() for m in mats.values()]),
                         q=0.95, minimum=5.0, step=1.0)

# Sized to keep type at 7 pt or more when the figure is printed 180 mm wide.
CHAPTER_FIG_W = 11.8
CHAPTER_ROW_H = 0.36
CHAPTER_FS_TICK = 13
CHAPTER_FS_RANGE = 12
CHAPTER_FS_LEGEND = 12.5
CHAPTER_FS_CBAR = 15
CHAPTER_RANGE_PAD = 6.5     # columns reserved for the share range at the right

mpl.rcParams["hatch.linewidth"] = 0.7
heights = [CHAPTER_ROW_H * m[0].shape[0] for m in mats.values()]
fig, axes = plt.subplots(len(mats), 1, figsize=(CHAPTER_FIG_W, sum(heights) + 1.2),
                         gridspec_kw={"height_ratios": heights})
axes = np.atleast_1d(axes)

for i, (age, (apc, sig, ok, cat)) in enumerate(mats.items()):
    ax = axes[i]
    draw_apc_heatmap(ax, apc, sig, ok, vmax=vmax_ch, cbar=False,
                     xlabels=(i == len(mats) - 1), fs_tick=CHAPTER_FS_TICK)
    draw_class_marks(ax, cat, ok)
    panel_label(ax, f"({panel_letter(i)}) {age}")
    ax.set_yticks(np.arange(apc.shape[0]) + 0.5)
    ax.set_yticklabels(apc.index, rotation=0, fontsize=CHAPTER_FS_TICK)
    # Range of the chapter's share of deaths across countries, beside each row.
    rng = chapter_share_range(chapters, age, apc.index.str.split(".").str[0])
    ax.set_xlim(0, apc.shape[1] + CHAPTER_RANGE_PAD)
    for r, (_, q) in enumerate(rng.iterrows()):
        if np.isfinite(q["min"]) and np.isfinite(q["max"]):
            ax.text(apc.shape[1] + 0.25, r + 0.5, f"{q['min']:.1f}-{q['max']:.1f}%",
                    va="center", ha="left", fontsize=CHAPTER_FS_RANGE, color="#495057")

plt.tight_layout(rect=[0.03, 0, 1, 0.985])
cax = fig.add_axes([1.015, 0.32, 0.012, 0.36])
sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(-vmax_ch, vmax_ch), cmap=CMAP_APC)
fig.colorbar(sm, cax=cax)
cax.set_ylabel("Annual change in mortality rate (%/year)", fontsize=CHAPTER_FS_CBAR,
               labelpad=6)
cax.tick_params(labelsize=CHAPTER_FS_RANGE)
fig.text(0.0, 0.5, "ICD-10 chapter", va="center", ha="center",
         rotation="vertical", fontsize=FS_AXIS)
fig.legend(handles=class_legend_handles(), loc="lower center",
           bbox_to_anchor=(0.5, 1.005), ncol=2, fontsize=CHAPTER_FS_LEGEND, frameon=True)
plt.tight_layout(rect=[0.03, 0, 1, 0.985])
savefig(fig, "Figure2_Chapter_APC_heatmap.png")
plt.show()

## Figure 3 — share of deaths by trend class

Percentage of each country's deaths, in each age group, falling in ICD-10 codes
of each class. Stagnation pools *significant but small* with *no material
change*. Age groups in which on average fewer than `FIG3_MIN_RESOLVED_PCT` of
deaths fall in a resolved class are not drawn.

In [ ]:
POOLED_OF = {
    "increasing": "increasing",
    "decreasing": "decreasing",
    "no material change": "stagnation",
    "significant but small": "stagnation",
    "inconclusive": "inconclusive",
    "not estimable": "not estimable",
}
POOLED_STACK = ["increasing", "stagnation", "inconclusive", "decreasing", "not estimable"]
POOLED_LABEL = {
    "increasing": f"Material increase ($\\geq${APC_THRESHOLD:g}%/year, FDR-significant)",
    "decreasing": f"Material decrease ($\\geq${APC_THRESHOLD:g}%/year, FDR-significant)",
    "stagnation": f"Stagnation (change below {APC_THRESHOLD:g}%/year)",
    "inconclusive": "Inconclusive",
    "not estimable": "Not estimable (too few deaths)",
}
POOLED_FILL = {
    "increasing": dict(color=COL_INC, hatch=None, edgecolor="none"),
    "decreasing": dict(color=COL_DEC, hatch=None, edgecolor="none"),
    "stagnation": dict(color=COL_NEU, hatch="..", edgecolor="#343a40"),
    "inconclusive": dict(color=COL_INCONCL, hatch=None, edgecolor="none"),
    "not estimable": dict(color=COL_NA, hatch="//", edgecolor="#ced4da"),
}
RESOLVED = ["increasing", "decreasing", "stagnation"]


def pooled_shares(df, age_group):
    d = df[df.age_group == age_group]
    tot = d.groupby("country", observed=True).total_deaths.sum()
    d = d.assign(klass=[POOLED_OF.get(c, "inconclusive") for c in d.category])
    return (d.groupby(["country", "klass"], observed=True).total_deaths.sum()
            .unstack(fill_value=0.0)
            .reindex(columns=POOLED_STACK, fill_value=0.0)
            .div(tot, axis=0).mul(100.0).dropna(how="all"))


resolved_pct = {a: float(pooled_shares(codes, a)[RESOLVED].sum(axis=1).mean())
                for a in panel_ages(codes)}
ages_p = [a for a, v in resolved_pct.items() if v >= FIG3_MIN_RESOLVED_PCT]
print("mean % of deaths in a resolved class:",
      {a: round(v, 1) for a, v in resolved_pct.items()})
order_p = [c for c in COUNTRIES
           if any(c in set(pooled_shares(codes, a).index) for a in ages_p)]

mpl.rcParams["hatch.linewidth"] = 0.7
fig, axes = plt.subplots(len(ages_p), 1,
                         figsize=(14, FIG3_PANEL_HEIGHT * len(ages_p) + 1.2), sharex=True)
axes = np.atleast_1d(axes)
for i, age in enumerate(ages_p):
    ax = axes[i]
    share = pooled_shares(codes, age).reindex(order_p).fillna(0.0)
    x = np.arange(len(share))
    bottom = np.zeros(len(share))
    for k in POOLED_STACK:
        vals = share[k].values
        ax.bar(x, vals, bottom=bottom, width=0.7, linewidth=0.0, **POOLED_FILL[k])
        bottom += vals
    ax.bar(x, 100, width=0.7, fill=False, edgecolor="#adb5bd", linewidth=0.5, zorder=5)
    ax.set_ylim(0, 100)
    ax.set_xlim(-0.5, len(share) - 0.5)
    ax.set_xticks(x)
    ax.set_yticks(np.arange(0, 101, 25))
    ax.tick_params(axis="y", labelsize=FS_TICK - 3)
    if i < len(ages_p) - 1:
        ax.set_xticklabels([])
        ax.tick_params(axis="x", length=0)
    else:
        ax.set_xticklabels(share.index, rotation=90, ha="center", va="top",
                           fontsize=FS_TICK - 3)
    ax.text(0.005, 0.97, f"({panel_letter(i)}) {age}", transform=ax.transAxes,
            va="top", ha="left", fontsize=FS_PANEL, fontweight="bold",
            backgroundcolor="white", alpha=0.8)

fig.text(0.02, 0.5, "Percentage of deaths (%)", va="center", ha="center",
         rotation="vertical", fontsize=FS_AXIS)
fig.legend(handles=[mpatches.Patch(label=POOLED_LABEL[k], facecolor=POOLED_FILL[k]["color"],
                                   hatch=POOLED_FILL[k]["hatch"],
                                   edgecolor=POOLED_FILL[k]["edgecolor"]
                                   if POOLED_FILL[k]["hatch"] else "#adb5bd")
                    for k in POOLED_STACK],
           loc="upper center", ncol=2, bbox_to_anchor=(0.55, 1.03),
           fontsize=13, framealpha=0.9)
plt.tight_layout(rect=[0.05, 0, 1, 0.95])
savefig(fig, "Figure3_TrendCategory_shares.png")
plt.show()

## Figure S1 — is a single straight line adequate over 2012–2019?

The fixed-knot test of Supplementary File 1, Section 2.5: against the
log-linear model, a change of slope at the midpoint of the window, fixed in
advance, compared by a quasi-likelihood $F$ test. The $p$-values are in the
estimates table (`lin_p_knot`); false discovery rate control is applied here
within each age group, as for the trends. The figure shows the all-cause series.

In [ ]:
def benjamini_hochberg(p):
    """BH adjusted p-values (the procedure of the estimation module)."""
    p = np.asarray(p, dtype=float)
    out = np.full(p.shape, np.nan)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return out
    pv = p[ok]
    m = pv.size
    order = np.argsort(pv)
    ranked = pv[order] * m / np.arange(1, m + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adj = np.empty(m)
    adj[order] = np.clip(ranked, 0, 1)
    out[ok] = adj
    return out


lin = all_cause[all_cause.estimable & all_cause.lin_p_knot.notna()].copy()
lin["q_knot"] = lin.groupby("age_group", observed=True).lin_p_knot.transform(
    lambda s: benjamini_hochberg(s.values))
lin["rejects"] = lin.q_knot.lt(0.05)

fig, ax = plt.subplots(figsize=(15.5, 5.6))
q = lin.pivot_table(index="age_group", columns="country", values="q_knot",
                    aggfunc="first", observed=False)
q = q.reindex([a for a in AGE_ORDER if a in q.index], columns=sorted(q.columns))
sns.heatmap(-np.log10(q.clip(lower=1e-4)), ax=ax, cmap="BuPu", vmin=0, vmax=4, cbar=True,
            cbar_kws={"label": r"$-\log_{10}$ (FDR-adjusted $p$)"},
            linewidths=0.4, linecolor="white")
cb = ax.collections[0].colorbar
cb.set_label(r"$-\log_{10}$ (FDR-adjusted $p$)", fontsize=15)
cb.ax.tick_params(labelsize=13)
for i, j in zip(*np.where(q.values < 0.05)):
    ax.add_patch(mpatches.Rectangle((j, i), 1, 1, fill=False,
                                    edgecolor="#212529", lw=1.6, zorder=3))
ax.invert_yaxis()
ax.set_xticks(np.arange(q.shape[1]) + 0.5)
ax.set_xticklabels(q.columns, rotation=90, fontsize=13)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=15)
ax.set_xlabel("")
ax.set_ylabel("Age group", fontsize=16)
ax.set_title("Outlined cells reject a single straight line over 2012-2019",
             loc="left", fontsize=17, fontweight="bold")
savefig(fig, "FigureS1_Linearity.png")
plt.show()

rej = lin[lin.rejects]
print(f"{len(rej)} of {len(lin)} estimable all-cause series reject a straight line, "
      f"across {rej.country.nunique()} countries; {int(rej.fdr_sig.sum())} of them "
      f"are classified as a material change")

### Comparison with the dated breakpoints

The breakpoints estimated by McCartney et al. (2022, appendix 6, Tables S6.1 and
S6.2) for the countries analysed here, set against the series that reject a
straight line. This reproduces the counts given in Supplementary File 1,
Section 2.5.

In [ ]:
YEAR0 = 2012.0
bp = pd.read_csv(BREAKPOINT_FILE)
col_le = next(c for c in bp.columns if "life expectancy" in c)
col_as = next(c for c in bp.columns if "ASMR" in c)

for label, col in (("Age-standardised mortality", col_as), ("Life expectancy", col_le)):
    have = bp[bp[col].notna()]
    early = int((have[col] <= YEAR0).sum())
    print(f"{label}: {len(have)} countries with a breakpoint, {early} "
          f"({100 * early / len(have):.0f}%) at or before {YEAR0:.0f}, "
          f"median {have[col].median():.1f}")

n_rej = rej.groupby("country").size()
dated = bp[bp[col_as].notna() | bp[col_le].notna()]
late = (dated[col_as] > YEAR0) | (dated[col_le] > YEAR0)
after, before = dated[late].Country, dated[~late].Country
for label, group in (("with a breakpoint after 2012 on either measure", after),
                     ("whose breakpoints all fall at or before 2012", before)):
    hit = sorted(c for c in group if c in n_rej.index)
    print(f"\n{len(group)} countries {label}; a straight line is rejected in "
          f"{len(hit)}: {', '.join(hit)}")
others = sorted(set(n_rej.index) - set(after) - set(before))
print("\nrejecting countries in neither group:",
      ", ".join(f"{c} ({', '.join(rej[rej.country == c].age_group.astype(str))})"
                for c in others))
print("\nseries rejecting, by country:")
print(n_rej.reindex(sorted(rej.country.unique())).to_string())

## Figure S2 — the primary estimate against the two comparison estimates

Left, the unadjusted trend (the same model with the age term suppressed);
right, the slope of the log ESP2013 directly standardised rate.

In [ ]:
ALT = [("crude_apc", "Crude: same model, age term suppressed"),
       ("dsr_apc", "ESP2013 directly standardised")]

cmp_df = all_cause[all_cause.estimable].dropna(subset=["apc", "crude_apc", "dsr_apc"]).copy()
ages_present = [a for a in AGE_ORDER if a in set(cmp_df.age_group.dropna())]
age_colour = {a: plt.cm.viridis(i / max(len(ages_present) - 1, 1))
              for i, a in enumerate(ages_present)}

lim = np.nanmax(np.abs(cmp_df[["apc", "crude_apc", "dsr_apc"]].values)) * 1.05
fig, axes = plt.subplots(1, 2, figsize=(13.2, 7.2), sharex=True, sharey=True)
for ax, (col, title) in zip(axes, ALT):
    ax.plot([-lim, lim], [-lim, lim], color="#868e96", lw=1.0, ls="--", zorder=1,
            label="equality")
    ax.axhline(0, color="#dee2e6", lw=0.8, zorder=0)
    ax.axvline(0, color="#dee2e6", lw=0.8, zorder=0)
    for a in ages_present:
        d = cmp_df[cmp_df.age_group == a]
        ax.scatter(d[col], d.apc, s=34, color=age_colour[a], edgecolor="white",
                   linewidth=0.5, zorder=3, label=a)
    r = cmp_df.apc.corr(cmp_df[col])
    mad = (cmp_df.apc - cmp_df[col]).abs().median()
    ax.set_title(title, loc="left", fontsize=13, fontweight="bold", pad=10)
    ax.text(0.03, 0.97, f"r = {r:.3f}\nmedian |difference| = {mad:.2f} %/year",
            transform=ax.transAxes, va="top", ha="left", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#dee2e6"))
    ax.set_xlabel("Alternative estimate (%/year)", fontsize=12)
    ax.set_aspect("equal")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.grid(color="#f1f3f5", lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
axes[0].set_ylabel("Age-adjusted estimate, this paper (%/year)", fontsize=12)

# Label the cells where the crude and adjusted estimates differ in sign.
flip = cmp_df[(cmp_df.apc * cmp_df.crude_apc < 0)].copy()
flip["gap"] = (flip.apc - flip.crude_apc).abs()
offsets = [(16, -10), (16, -30), (16, 14), (-120, -26)]
for (dx, dy), (_, r_) in zip(offsets, flip.nlargest(4, "gap").iterrows()):
    axes[0].annotate(f"{r_.country} {r_.age_group}", xy=(r_.crude_apc, r_.apc),
                     xytext=(dx, dy), textcoords="offset points", fontsize=9.5,
                     color="#212529",
                     arrowprops=dict(arrowstyle="-", lw=0.7, color="#868e96"))

h, l = axes[0].get_legend_handles_labels()
h = [plt.Line2D([], [], ls="none")] + h
l = ["Age group:"] + l
fig.legend(h, l, loc="lower center", ncol=len(ages_present) + 2, fontsize=11,
           frameon=True, bbox_to_anchor=(0.5, 0.002), handletextpad=0.5,
           columnspacing=1.1)
fig.tight_layout(rect=[0, 0.09, 1, 1])
savefig(fig, "FigureS2_Estimator_comparison.png")
plt.show()

## Figure S3 — mortality level by country and age group

Each country's ESP2013 directly standardised rate at mid-period (the geometric
mean of the first and last year), as a ratio to the median country of the same
age group.

In [ ]:
LEVEL_RATIO_CMAP = LinearSegmentedColormap.from_list(
    "level_ratio", ["#1b7837", "#a6dba0", "#f7f7f7", "#c2a5cf", "#762a83"])

lv = all_cause[["country", "age_group", "dsr_esp2013_first", "dsr_esp2013_last",
                "status"]].copy()
lv["rate_mid"] = np.sqrt(lv.dsr_esp2013_first * lv.dsr_esp2013_last)
lv = lv[lv.rate_mid.gt(0) & np.isfinite(lv.rate_mid)]
med = lv.groupby("age_group", observed=True).rate_mid.median()
lv["ratio"] = lv.rate_mid / lv.age_group.astype(str).map(med.rename(index=str)).astype(float)
mat = lv.pivot_table(index="age_group", columns="country", values="ratio",
                     observed=True).reindex(
    index=[a for a in AGE_ORDER if a in set(lv.age_group)], columns=COUNTRIES)

lim = min(float(np.nanmax(np.abs(np.log2(mat.values)))), 1.2)

fig, ax = plt.subplots(figsize=(max(12, 0.42 * len(COUNTRIES)), 5.6))
cax = fig.add_axes([1.005, 0.14, 0.014, 0.72])
sns.heatmap(np.log2(mat), ax=ax, cmap=LEVEL_RATIO_CMAP, vmin=-lim, vmax=lim,
            center=0, cbar=True, cbar_ax=cax, linewidths=0.4, linecolor="white")
ax.set_facecolor(COL_NA)
for i, j in zip(*np.where(mat.isna().values)):
    ax.add_patch(mpatches.Rectangle((j, i), 1, 1, fill=True, facecolor=COL_NA,
                                    hatch="////", edgecolor="#adb5bd",
                                    linewidth=0.0, zorder=3))
ax.invert_yaxis()
ax.set_yticklabels([f"{a}  (median {med[a]:,.0f})" for a in mat.index],
                   rotation=0, fontsize=11)
ax.set_xticklabels(mat.columns, rotation=90, fontsize=9)
ax.set_ylabel("Age group", fontsize=FS_LABEL)
ax.set_xlabel("")
ticks = [-1, -np.log2(1.5), 0, np.log2(1.5), 1]
cax.set_yticks(ticks)
cax.set_yticklabels(["0.5x", "0.67x", "median", "1.5x", "2x"], fontsize=11)
cax.set_ylabel("Rate relative to the age-group median", fontsize=12, labelpad=8)
savefig(fig, "FigureS3_Level_relative_to_median.png")
plt.show()

## Figure S4 — all-cause trends with confidence intervals, every age group

One country order, taken from the ages 30–44 estimates, shared by every panel.
A dotted ring marks an estimate whose interval excludes zero but which does not
survive false discovery rate control.

In [ ]:
DOTTED_RING_STYLE = (0, (1.2, 1.8))


class HandlerDottedRing(HandlerPatch):
    """Legend key for the dotted ring, drawn as a patch so the dots show."""
    def create_artists(self, legend, orig_handle, xdescent, ydescent,
                       width, height, fontsize, trans):
        r = 0.5 * min(width, height)
        ring = mpatches.Circle((width / 2 - xdescent, height / 2 - ydescent), r,
                               facecolor="none", edgecolor=orig_handle.get_edgecolor(),
                               linewidth=orig_handle.get_linewidth(),
                               linestyle=DOTTED_RING_STYLE)
        ring.set_transform(trans)
        return [ring]


class DotCircle(mpatches.Circle):
    """Legend key for 'inconclusive': a filled disc with a centre dot."""


class HandlerDotCircle(HandlerPatch):
    def create_artists(self, legend, orig_handle, xdescent, ydescent,
                       width, height, fontsize, trans):
        cx, cy = width / 2 - xdescent, height / 2 - ydescent
        r = 0.5 * min(width, height)
        disc = mpatches.Circle((cx, cy), r, facecolor=orig_handle.get_facecolor(),
                               edgecolor=orig_handle.get_edgecolor(), linewidth=0.6)
        dot = mpatches.Circle((cx, cy), 0.30 * r, facecolor="#495057", edgecolor="none")
        for a in (disc, dot):
            a.set_transform(trans)
        return [disc, dot]


LEGEND_HANDLERS = {mpatches.Circle: HandlerDottedRing(), DotCircle: HandlerDotCircle()}

FOREST_ORDER_AGE = "30-44"
FOREST_XLIM = (-10.0, 10.0)

ages_all = panel_ages(all_cause)
ref = (all_cause[all_cause.age_group.eq(FOREST_ORDER_AGE) & all_cause.estimable]
       .sort_values("apc", ascending=False))
country_order = list(ref.country) + [c for c in COUNTRIES if c not in set(ref.country)]

fig, axes = plt.subplots(1, len(ages_all), sharey=True,
                         figsize=(2.55 * len(ages_all) + 1.6,
                                  0.235 * len(country_order) + 1.9))
axes = np.atleast_1d(axes).ravel()
y = np.arange(len(country_order))[::-1]          # first country at the top
ypos = dict(zip(country_order, y))
xlo, xhi = FOREST_XLIM
for k, age in enumerate(ages_all):
    ax = axes[k]
    d = all_cause[all_cause.age_group.eq(age)].set_index("country")
    ax.axvspan(-APC_THRESHOLD, APC_THRESHOLD, color="#f8f9fa", zorder=0)
    ax.axvline(0, color="#495057", lw=0.9, zorder=1)
    for c in country_order:
        yy = ypos[c]
        if c not in d.index or not bool(d.at[c, "estimable"]):
            ax.axhspan(yy - 0.42, yy + 0.42, color="#f1f3f5", zorder=0)
            continue
        r = d.loc[c]
        ax.hlines(yy, max(r.apc_lo, xlo), min(r.apc_hi, xhi), color="#868e96",
                  lw=1.2, zorder=2)
        if r.apc_lo < xlo:
            ax.plot(xlo, yy, marker="<", ms=4, color="#868e96", zorder=2)
        if r.apc_hi > xhi:
            ax.plot(xhi, yy, marker=">", ms=4, color="#868e96", zorder=2)
        ax.scatter(r.apc, yy, s=30, c=[category_colour(r.category, r.apc)],
                   edgecolor="#343a40", linewidth=0.5, zorder=3)
        if r.category == "inconclusive":
            ax.scatter(r.apc, yy, s=3.5, c="#495057", linewidths=0, zorder=4)
        if ((r.apc_lo > 0) or (r.apc_hi < 0)) and r.p_bh >= 0.05:
            ax.scatter(r.apc, yy, s=115, facecolor="none", edgecolor="#495057",
                       linewidth=1.0, linestyle=DOTTED_RING_STYLE, zorder=5)
    ax.set_xlim(xlo, xhi)
    ax.set_ylim(-0.7, len(country_order) - 0.3)
    ax.set_title(f"({panel_letter(k)}) {age}", loc="left", fontsize=12, fontweight="bold")
    ax.set_xticks([xlo, xlo / 2, 0, xhi / 2, xhi])
    ax.set_xticklabels([f"{t:g}" for t in (xlo, xlo / 2, 0, xhi / 2, xhi)])
    ax.tick_params(axis="x", labelsize=10)
    ax.grid(axis="x", color="#e9ecef", lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)

axes[0].set_yticks(y)
axes[0].set_yticklabels(country_order, fontsize=9.5)
axes[0].spines["left"].set_visible(True)
fig.supxlabel("Annual change in all-cause mortality rate (%/year)", fontsize=13, y=0.01)

cats = ["increasing", "significant but small (increase)",
        "significant but small (decrease)", "inconclusive",
        "no material change", "decreasing"]
handles, labels = [], []
for c in cats:
    if c == "inconclusive":
        h = DotCircle((0.5, 0.5), 0.5, facecolor=CATEGORY_COLOURS[c],
                      edgecolor="#343a40", linewidth=0.6)
    else:
        h = plt.Line2D([], [], marker="o", linestyle="none", markersize=9,
                       markerfacecolor=CATEGORY_COLOURS[c],
                       markeredgecolor="#343a40", markeredgewidth=0.5)
    handles.append(h)
    labels.append(c.replace("significant but small", "signif. but small").capitalize())
handles.append(mpatches.Circle((0.5, 0.5), 0.5, facecolor="none", edgecolor="#495057",
                               linewidth=1.0, linestyle=DOTTED_RING_STYLE))
labels.append("Interval excludes zero, but not after FDR control")
fig.legend(handles=handles, labels=labels, loc="upper center", ncol=4, fontsize=10.5,
           bbox_to_anchor=(0.5, 1.015), frameon=True, handler_map=LEGEND_HANDLERS)
fig.tight_layout(rect=[0, 0.015, 1, 0.945])
savefig(fig, "FigureS4_AllCause_forest_all_ages.png")
plt.show()

## Figure S5 — detectability

The smallest annual change each all-cause series could have detected with 80%
power, $\mathrm{MDE}_{80} = 100\{\exp(2.8016\,\mathrm{se}(\hat\beta)) - 1\}$.
Outlined cells could not have detected a change of `APC_THRESHOLD`.

In [ ]:
mde = pivot(all_cause, "mde80", countries=COUNTRIES)
mde_plot = mde.clip(upper=np.nanquantile(mde.values, 0.98))

fig, ax = plt.subplots(figsize=(max(12, 0.42 * len(COUNTRIES)), 5.6))
sns.heatmap(mde_plot, ax=ax, cmap="YlOrBr", cbar=True,
            cbar_kws={"label": "Minimum detectable change (%/year, 80% power)"},
            linewidths=0.4, linecolor="white")
for i, j in zip(*np.where(mde.values > APC_THRESHOLD)):
    ax.add_patch(mpatches.Rectangle((j, i), 1, 1, fill=False,
                                    edgecolor="#212529", lw=1.4, zorder=3))
ax.invert_yaxis()
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=FS_TICK)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=FS_TICK)
ax.set_ylabel("Age group", fontsize=FS_LABEL)
ax.set_xlabel("")
ax.legend(handles=[mpatches.Patch(facecolor="none", edgecolor="#212529",
                                  label=f"Cannot detect a change of {APC_THRESHOLD:g} %/year")],
          loc="lower center", bbox_to_anchor=(0.5, 1.04), fontsize=12, frameon=True)
savefig(fig, "FigureS5_Detectability.png")
plt.show()

## Figure S6 — estimate against its standard error, ICD-10 code level

In [ ]:
d = codes[codes.estimable & codes.se.notna() & codes.apc.notna()].copy()
d = d[np.isfinite(d.apc) & (d.se > 0)]

funnel_ages = panel_ages(d)
ncols = min(4, len(funnel_ages))
nrows = int(np.ceil(len(funnel_ages) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 4.9 * nrows), sharey=True)
flat = np.atleast_1d(axes).ravel()

for i, age in enumerate(funnel_ages):
    ax = flat[i]
    dd = d[d.age_group == age]
    rest = dd[~dd.fdr_sig]
    inc = dd[dd.fdr_sig & (dd.apc > 0)]
    dec = dd[dd.fdr_sig & (dd.apc < 0)]
    ax.scatter(rest.se * 100, rest.apc, s=7, c="#ced4da", alpha=0.55,
               linewidths=0, label="Not significant")
    ax.scatter(dec.se * 100, dec.apc, s=14, c=COL_DEC, linewidths=0,
               label="Decreasing (FDR)")
    ax.scatter(inc.se * 100, inc.apc, s=14, c=COL_INC, linewidths=0,
               label="Increasing (FDR)")
    xs = np.linspace(max(dd.se.min() * 100, 1e-3), dd.se.quantile(0.995) * 100, 200)
    ax.plot(xs, 1.96 * xs, ls="--", c="#495057", lw=1.0)
    ax.plot(xs, -1.96 * xs, ls="--", c="#495057", lw=1.0)
    ax.axhline(0, color="#495057", lw=0.8)
    ax.set_xscale("log")
    ax.set_ylim(-40, 40)
    ax.set_title(f"({panel_letter(i)}) {age}", loc="left", fontsize=15, fontweight="bold")
    if i >= len(funnel_ages) - ncols:
        ax.set_xlabel("Standard error of the trend (%/year, log scale)", fontsize=12)
    if i % ncols == 0:
        ax.set_ylabel("Annual change in mortality rate (%/year)", fontsize=13)
    ax.grid(color="#e9ecef", lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

spare = flat[len(funnel_ages):]
handles, labels = flat[0].get_legend_handles_labels()
if len(spare):
    spare[0].legend(handles, labels, fontsize=12, loc="center", frameon=True,
                    markerscale=2.0)
    for ax in spare:
        ax.axis("off")
else:
    flat[0].legend(fontsize=11, loc="upper left", frameon=True, markerscale=1.6)
fig.tight_layout()
savefig(fig, "FigureS6_Precision_funnel.png")
plt.show()

## Sex-specific trends

From the age-by-sex model of Supplementary File 1, Section 7: one model per cell
with a sex-by-year interaction, so that `apc_female` is $\beta$, `apc_male` is
$\beta+\gamma$ and `apc_diff` is $100(e^{\gamma}-1)$, the additional annual
percentage change in men relative to women.

In [ ]:
COL_MALE = "#7048e8"      # violet
COL_FEMALE = "#e8590c"    # orange
COL_DIFF_NS = "#ced4da"


def load_sex_estimates(path):
    df = pd.read_csv(path, low_memory=False)
    df = df[df.country.isin(ANALYSIS_COUNTRIES)].copy()
    df["age_group"] = pd.Categorical(df.age_group, categories=AGE_ORDER, ordered=True)
    df = df[df.age_group.notna()]
    df["estimable"] = df.status.isin(ESTIMABLE)
    for which in ("male", "female", "sexavg", "diff"):
        q = f"apc_{which}_p_bh"
        if q in df.columns:
            df[f"sig_{which}"] = (df[q].lt(0.05) & df[f"apc_{which}"].abs().ge(APC_THRESHOLD)
                                  & df.estimable)
    print(f"{Path(path).name}: {len(df)} rows, {df.country.nunique()} countries")
    return df


sex_all = load_sex_estimates(EST_SEX_ALL_CAUSE)
sex_chapters = load_sex_estimates(EST_SEX_CHAPTER)
SEX_COUNTRIES = sorted(sex_all.country.unique())
SEX_AGES = [a for a in AGE_ORDER if a in set(sex_all.age_group.dropna())]

### Figure S7 — men, women and the difference

In [ ]:
def sex_matrices(df, value, flag, countries, ages):
    apc = df.pivot_table(index="age_group", columns="country", values=value,
                         aggfunc="first", observed=False)
    sig = df.pivot_table(index="age_group", columns="country", values=flag,
                         aggfunc="first", observed=False)
    ok = df.pivot_table(index="age_group", columns="country", values="estimable",
                        aggfunc="first", observed=False)
    idx = [a for a in ages if a in apc.index]
    cols = [c for c in countries if c in apc.columns]
    return (apc.reindex(idx, columns=cols), sig.reindex(idx, columns=cols),
            ok.reindex(idx, columns=cols))


panels = [("apc_male", "sig_male", "(a) Males"),
          ("apc_female", "sig_female", "(b) Females"),
          ("apc_diff", "sig_diff", "(c) Difference: additional annual change in men")]
smats = {lab: sex_matrices(sex_all, v, f, SEX_COUNTRIES, SEX_AGES) for v, f, lab in panels}
vmax_sex = symmetric_vmax(
    np.concatenate([smats[lab][0].values.ravel() for _, _, lab in panels[:2]]), q=0.95)
vmax_diff = symmetric_vmax(smats[panels[2][2]][0].values.ravel(), q=0.95)

h = 0.52 * len(SEX_AGES) + 1.0
fig, axes = plt.subplots(3, 1, figsize=(max(12, 0.42 * len(SEX_COUNTRIES)), 3 * h))
cax1 = fig.add_axes([1.005, 0.42, 0.012, 0.44])
cax2 = fig.add_axes([1.005, 0.08, 0.012, 0.22])
for i, (value, flag, label) in enumerate(panels):
    apc, sig, ok = smats[label]
    draw_apc_heatmap(axes[i], apc, sig, ok, vmax=(vmax_diff if i == 2 else vmax_sex),
                     cbar=(i in (0, 2)),
                     cbar_ax=(cax1 if i == 0 else (cax2 if i == 2 else None)),
                     xlabels=(i == 2), fs_tick=13)
    axes[i].invert_yaxis()
    axes[i].set_ylabel("Age group", fontsize=14)
    panel_label(axes[i], label, fs=15)
cax1.set_ylabel("Annual change (%/year)", fontsize=11)
cax2.set_ylabel("Men minus women (%/year)", fontsize=11)
for c in (cax1, cax2):
    c.tick_params(labelsize=10)
axes[0].legend(handles=dot_legend_handles(), loc="lower center",
               bbox_to_anchor=(0.5, 1.12), ncol=2, fontsize=11, frameon=True)
plt.tight_layout()
savefig(fig, "FigureS7_Sex_APC_heatmaps.png")
plt.show()

### Figure S8 — both sexes with intervals

One country order, taken from the difference at ages 30–44, shared by every
panel. Each panel is scaled to the central range of its own intervals.

In [ ]:
def sex_dumbbell(ax, d, title, order, xlim, show_ylabels, fs_tick=9.5):
    d = d[d.estimable & d.apc_male.notna() & d.apc_female.notna()].copy()
    d = d.set_index("country").reindex(list(order)).reset_index()
    y = np.arange(len(d))[::-1]
    for yi, ok in zip(y, d.apc_male.notna() & d.apc_female.notna()):
        if not ok:
            ax.axhspan(yi - 0.45, yi + 0.45, color="#f1f3f5", zorder=0)
    d = d.assign(_y=y)
    d = d[d.apc_male.notna() & d.apc_female.notna()]
    y = d._y.values
    for yi, (_, r) in zip(y, d.iterrows()):
        sig = bool(r.get("sig_diff", False))
        ax.plot([r.apc_female, r.apc_male], [yi, yi],
                color="#495057" if sig else COL_DIFF_NS, lw=2.6 if sig else 1.4,
                zorder=2, solid_capstyle="round")
    ax.hlines(y, d.apc_female_lo, d.apc_female_hi, color=COL_FEMALE, lw=0.9,
              alpha=0.55, zorder=1)
    ax.hlines(y, d.apc_male_lo, d.apc_male_hi, color=COL_MALE, lw=0.9,
              alpha=0.55, zorder=1)
    ax.scatter(d.apc_female, y, s=42, c=COL_FEMALE, edgecolor="white",
               linewidth=0.6, zorder=3, label="Women")
    ax.scatter(d.apc_male, y, s=42, c=COL_MALE, edgecolor="white",
               linewidth=0.6, zorder=3, label="Men")
    ax.axvline(0, color="#495057", lw=1.0, zorder=0)
    ax.set_yticks(np.arange(len(order))[::-1])
    if show_ylabels:
        ax.set_yticklabels(list(order), fontsize=fs_tick)
    else:
        ax.tick_params(axis="y", labelleft=False)
    ax.set_ylim(-0.8, len(order) - 0.2)
    ax.set_xlim(*xlim)
    ax.tick_params(axis="x", labelsize=11)
    ax.set_title(title, loc="left", fontsize=14, fontweight="bold")
    ax.grid(axis="x", color="#dee2e6", lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def dumbbell_xlim(d, q=0.05, pad=0.08):
    lo = min(np.nanquantile(d.apc_female_lo, q), np.nanquantile(d.apc_male_lo, q))
    hi = max(np.nanquantile(d.apc_female_hi, 1 - q), np.nanquantile(d.apc_male_hi, 1 - q))
    m = pad * (hi - lo)
    return (lo - m, hi + m)


show_ages = panel_ages(sex_all)
shown = sex_all[sex_all.age_group.isin(show_ages) & sex_all.estimable
                & sex_all.apc_male.notna() & sex_all.apc_female.notna()]
XLIMS = {a: dumbbell_xlim(shown[shown.age_group == a]) for a in show_ages}
_ref = sex_all[sex_all.age_group.eq("30-44") & sex_all.estimable]
SEX_COUNTRY_ORDER = (list(_ref.sort_values("apc_diff", ascending=False).country)
                     + [c for c in SEX_COUNTRIES if c not in set(_ref.country)])

ncols = len(show_ages)
fig, axes = plt.subplots(1, ncols, sharey=True, squeeze=False,
                         figsize=(2.75 * ncols + 1.6, 0.235 * len(SEX_COUNTRY_ORDER) + 1.9))
for i, age in enumerate(show_ages):
    sex_dumbbell(axes[0][i], sex_all[sex_all.age_group == age],
                 f"({panel_letter(i)}) {age}", SEX_COUNTRY_ORDER, XLIMS[age],
                 show_ylabels=(i == 0))
fig.supxlabel("Annual change in all-cause mortality rate (%/year)",
              fontsize=FS_LABEL, y=-0.01)
handles = [
    plt.Line2D([], [], marker="o", color="none", markerfacecolor=COL_MALE,
               markersize=9, label="Men"),
    plt.Line2D([], [], marker="o", color="none", markerfacecolor=COL_FEMALE,
               markersize=9, label="Women"),
    plt.Line2D([], [], color="#495057", lw=2.6,
               label="Difference significant after FDR control"),
    plt.Line2D([], [], color=COL_DIFF_NS, lw=1.4, label="Difference not resolved"),
]
fig.legend(handles=handles, loc="upper center", ncol=4, fontsize=11,
           bbox_to_anchor=(0.5, 1.012), frameon=True)
fig.tight_layout()
savefig(fig, "FigureS8_Sex_dumbbell.png")
plt.show()

### Figure S9 — how much of the sex comparison is resolvable

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.6))

ax = axes[0]
d = sex_all[sex_all.estimable & sex_all.apc_male.notna() & sex_all.apc_female.notna()]
lims = [min(d.apc_female.min(), d.apc_male.min()) - 0.5,
        max(d.apc_female.max(), d.apc_male.max()) + 0.5]
ax.plot(lims, lims, ls="--", c="#adb5bd", lw=1.0, zorder=0)
ax.axhline(0, c="#dee2e6", lw=0.8, zorder=0)
ax.axvline(0, c="#dee2e6", lw=0.8, zorder=0)
markers = {"0": "o", "1-14": "s", "15-29": "^", "30-44": "D",
           "45-59": "v", "60-74": "P", "75+": "X"}
for age in SEX_AGES:
    dd = d[d.age_group == age]
    sig = dd.sig_diff.fillna(False).astype(bool)
    ax.scatter(dd.apc_female[~sig], dd.apc_male[~sig], s=34, marker=markers[age],
               facecolor="none", edgecolor="#adb5bd", linewidth=0.8, zorder=2)
    ax.scatter(dd.apc_female[sig], dd.apc_male[sig], s=52, marker=markers[age],
               alpha=0.9, edgecolor="#343a40", linewidth=0.5, zorder=3, label=age)
for _, r in d.assign(g=d.apc_diff.abs()).nlargest(5, "g").iterrows():
    ax.annotate(f"{r.country} {r.age_group}", (r.apc_female, r.apc_male),
                textcoords="offset points", xytext=(7, 5), fontsize=9, color="#495057",
                arrowprops=dict(arrowstyle="-", color="#ced4da", lw=0.7))
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Annual change, women (%/year)", fontsize=13)
ax.set_ylabel("Annual change, men (%/year)", fontsize=13)
ax.set_title("(a) Men against women; filled = difference resolved",
             loc="left", fontsize=14, fontweight="bold")
ax.legend(fontsize=9, title="Age group", title_fontsize=9, loc="lower right", ncol=2)
ax.grid(color="#f1f3f5", lw=0.6)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)

ax = axes[1]
mde = sex_all.pivot_table(index="age_group", columns="country", values="apc_diff_mde80",
                          aggfunc="first", observed=False)
mde = mde.reindex(SEX_AGES, columns=SEX_COUNTRIES)
mde_plot = mde.clip(upper=np.nanquantile(mde.values, 0.95))
sns.heatmap(mde_plot, ax=ax, cmap="YlOrBr", cbar=True,
            cbar_kws={"label": "Minimum detectable difference (%/year, 80% power)"},
            linewidths=0.4, linecolor="white")
for i, j in zip(*np.where(mde.values > APC_THRESHOLD)):
    ax.add_patch(mpatches.Rectangle((j, i), 1, 1, fill=False, edgecolor="#212529",
                                    lw=1.1, zorder=3))
ax.invert_yaxis()
ax.set_xticks(np.arange(mde_plot.shape[1]) + 0.5)
ax.set_xticklabels(mde_plot.columns, rotation=90, fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11)
ax.set_xlabel("")
ax.set_ylabel("Age group", fontsize=13)
ax.set_title(f"(b) Outlined: cannot resolve a difference of {APC_THRESHOLD:g} %/year",
             loc="left", fontsize=14, fontweight="bold")
fig.tight_layout()
savefig(fig, "FigureS9_Sex_resolvability.png")
plt.show()

### Figure S10 — sex differences by ICD-10 chapter

Age groups with at least one chapter-by-country cell whose sex difference is
significant after false discovery rate control and at least `APC_THRESHOLD`;
within them, the chapters holding such a cell.

In [ ]:
def sex_chapter_matrices(df, age_group, countries, min_share_pct=MIN_CHAPTER_SHARE_PCT):
    d = df[df.age_group == age_group]
    share = d.groupby("cause", observed=True).share_of_group_deaths_pct.mean()
    d = d[d.cause.isin(share[share >= min_share_pct].index)]
    m = {v: d.pivot_table(index="cause", columns="country", values=v,
                          aggfunc="first", observed=False)
         for v in ("apc_diff", "sig_diff", "estimable")}
    rows = [c for c in CHAPTER_ROMAN if c in m["apc_diff"].index]
    cols = [c for c in countries if c in m["apc_diff"].columns]
    m = {k: v.reindex(rows, columns=cols) for k, v in m.items()}
    keep = m["sig_diff"].fillna(False).astype(bool).any(axis=1)
    keep = keep[keep].index
    return tuple(m[k].loc[keep].rename(index=nice_chapter)
                 for k in ("apc_diff", "sig_diff", "estimable"))


res = sex_chapters[sex_chapters.sig_diff.fillna(False).astype(bool)]
ages_ch = [a for a in AGE_ORDER if a in set(res.age_group.dropna())]
print("age groups shown:", ages_ch)
cmats = {a: sex_chapter_matrices(sex_chapters, a, SEX_COUNTRIES) for a in ages_ch}
vmax_cd = symmetric_vmax(np.concatenate([m[0].values.ravel() for m in cmats.values()]),
                         q=0.95, minimum=5.0, step=1.0)

heights = [max(2.0, 0.46 * m[0].shape[0]) for m in cmats.values()]
fig, axes = plt.subplots(len(ages_ch), 1,
                         figsize=(max(16, 0.5 * len(SEX_COUNTRIES)), sum(heights) + 1.5),
                         gridspec_kw={"height_ratios": heights})
axes = np.atleast_1d(axes)
cax = fig.add_axes([1.005, 0.30, 0.011, 0.40])
for i, (age, (apc, sig, ok)) in enumerate(cmats.items()):
    draw_apc_heatmap(axes[i], apc, sig, ok, vmax=vmax_cd, cbar=(i == 0),
                     cbar_ax=cax if i == 0 else None,
                     xlabels=(i == len(ages_ch) - 1), fs_tick=13)
    panel_label(axes[i], f"({panel_letter(i)}) {age}")
cax.set_ylabel("Additional annual change in men (%/year)", fontsize=12)
cax.tick_params(labelsize=10)
fig.text(0.0, 0.5, "ICD-10 chapter", va="center", ha="center",
         rotation="vertical", fontsize=FS_AXIS)
axes[0].legend(handles=dot_legend_handles(), loc="lower center",
               bbox_to_anchor=(0.5, 1.12), ncol=2, fontsize=11, frameon=True)
plt.tight_layout(rect=[0.03, 0, 1, 0.99])
savefig(fig, "FigureS10_Sex_chapter_differences.png")
plt.show()

## Supplementary File 2 — workbook of estimates

Formatted `APC (95% CI)` tables and class summaries for the combined-sex
analysis, followed by the sex-specific sheets. The `Readme` sheet describes each
sheet.

In [ ]:
def fmt_ci(r):
    if not np.isfinite(r.apc) or not np.isfinite(r.apc_lo):
        return "not estimable"
    star = "*" if r.p_bh < 0.05 else ""
    return f"{r.apc:+.2f} ({r.apc_lo:+.2f}, {r.apc_hi:+.2f}){star}"


def supp_table(df, index, columns="country"):
    t = df.copy()
    t["cell"] = t.apply(fmt_ci, axis=1)
    return t.pivot_table(index=index, columns=columns, values="cell",
                         aggfunc="first", observed=False)


# The two intermediate classes are reported together as stagnation, a claim
# about magnitude rather than significance. Figures 1 and 2 keep them apart
# (open circle, cross-hatching) because the evidence behind them differs.
CLASS_OF = {
    "increasing": "Increasing",
    "decreasing": "Decreasing",
    "no material change": "Stagnation",
    "significant but small": "Stagnation",
    "inconclusive": "Inconclusive",
    "not estimable": "Not estimable",
}
CLASS_COLS = ["Increasing", "Decreasing", "Stagnation",
              "Inconclusive", "Not estimable"]
PCT_OF = {"Increasing": "% deaths increasing",
          "Decreasing": "% deaths decreasing",
          "Stagnation": "% deaths stagnating",
          "Inconclusive": "% deaths inconclusive",
          "Not estimable": "% deaths not estimable"}


def _classified(df):
    return df.assign(klass=df.category.map(CLASS_OF).fillna("Inconclusive"))


def chapter_class_counts(df):
    """ICD-10 chapters in each class, by country and age group.

    Two quantities, side by side, because they can tell opposite stories: the
    NUMBER of chapters in each class, and the SHARE OF DEATHS those chapters
    carry. A country can have a third of its chapters rising while the rising
    chapters hold a fraction of its deaths, if the large chapters - neoplasms
    and circulatory disease at the older ages - are not among them.

    The death-weighted columns are the chapter-level counterpart of the
    "Death share by class" sheet, which is computed at ICD-10 code level. The
    two still differ, because a chapter is a single fit on pooled counts and
    has far more power than any of the codes inside it; see the Readme.

    Every chapter present in the table is counted, with no minimum-share
    filter: that filter governs what Figure 2 displays, not what is recorded.
    """
    c = _classified(df)
    n = (c.groupby(["age_group", "country", "klass"], observed=True)
         .cause.nunique().unstack(fill_value=0)
         .reindex(columns=CLASS_COLS, fill_value=0))
    n["Chapters assessed"] = n.sum(axis=1)

    tot = c.groupby(["age_group", "country"], observed=True).total_deaths.sum()
    w = (c.groupby(["age_group", "country", "klass"], observed=True)
         .total_deaths.sum().unstack(fill_value=0.0)
         .reindex(columns=CLASS_COLS, fill_value=0.0))
    w = w.div(tot, axis=0).mul(100.0).round(2)
    w.columns = [PCT_OF[k] for k in CLASS_COLS]
    w["Total deaths"] = tot.astype("int64")

    return n.join(w).reset_index().sort_values(["age_group", "country"])


def death_share_by_class(df):
    """Percentage of deaths in each class, by country and age group."""
    c = _classified(df)
    tot = c.groupby(["age_group", "country"], observed=True).total_deaths.sum()
    t = (c.groupby(["age_group", "country", "klass"], observed=True)
         .total_deaths.sum().unstack(fill_value=0.0)
         .reindex(columns=CLASS_COLS, fill_value=0.0))
    t = t.div(tot, axis=0).mul(100.0).round(2)
    t["Total deaths"] = tot.astype("int64")
    return t.reset_index().sort_values(["age_group", "country"])


def codes_most_often_rising(df, min_countries=2):
    """ICD-10 codes ranked by how many countries show a material increase.

    "Material" is the classification of the main analysis: significant after
    Benjamini-Hochberg control AND at least APC_THRESHOLD per year. The last
    column names the countries, so the count can be checked against the
    estimates table.
    """
    rising = _classified(df)
    rising = rising[rising.klass == "Increasing"]
    assessed = (df[df.estimable].groupby(["age_group", "cause"], observed=True)
                .country.nunique().rename("Countries assessed"))
    # The pipeline carries no free-text description for an individual code -
    # cause_label falls back to the code itself - so the chapter is given
    # instead, read from the estimation module that defines the mapping.
    try:
        from mortality_estimates import icd10_chapter
    except Exception:
        icd10_chapter = None
    g = rising.groupby(["age_group", "cause"], observed=True)
    # The size of the cause matters as much as the count of countries: a code
    # can rise in twenty countries and still account for a twentieth of a
    # percent of deaths. Both shares are given so that a reader can weigh them.
    overall = (df[df.estimable].groupby(["age_group", "cause"], observed=True)
               .share_of_group_deaths_pct.mean().round(3)
               .rename("Mean % of deaths, all countries"))
    out = pd.DataFrame({
        "Countries rising": g.country.nunique(),
        "Mean % of deaths where rising": g.share_of_group_deaths_pct.mean().round(2),
        "Deaths in rising cells": g.total_deaths.sum().astype("int64"),
        "Countries": g.country.apply(lambda s: ", ".join(sorted(s))),
    }).join(assessed).join(overall)
    out = out[out["Countries rising"] >= min_countries].reset_index()
    if icd10_chapter is not None:
        ch = {c: icd10_chapter(c) for c in out.cause.unique()}
        out.insert(2, "ICD-10 chapter",
                   out.cause.map(lambda c: ". ".join(x for x in ch[c] if x)))
    out = out.rename(columns={"age_group": "Age group", "cause": "ICD-10 code"})
    return out.sort_values(["Age group", "Countries rising",
                            "Mean % of deaths, all countries"],
                           ascending=[True, False, False])


README = pd.DataFrame({"": [
    "Supplementary estimates accompanying the revised manuscript.",
    "",
    "Every trend is the annual percentage change in the mortality rate from a",
    "quasi-Poisson model with a log link, log person-years as offset and an",
    "indicator for each age unit within the broad band, so trends are adjusted",
    "for age composition within the band. See Supplementary File 1 for the full",
    "specification.",
    "",
    "Cells are placed in one of five classes, as in Figures 1 and 2:",
    f"  Increasing / Decreasing  change of at least {APC_THRESHOLD:g}%/year that survives",
    "                           Benjamini-Hochberg control of the false discovery",
    "                           rate within its family of tests",
    f"  Stagnation               change estimated below {APC_THRESHOLD:g}%/year: either the",
    "                           estimate survives the same control while falling",
    "                           below the threshold, or the confidence interval",
    "                           lies entirely within it. Figures 1 and 2 keep the",
    "                           two apart, with an open circle and cross-hatching",
    "                           respectively, because the evidence differs.",
    "  Inconclusive             the deaths available cannot resolve a change of",
    "                           that size. This is an absence of evidence, NOT",
    "                           evidence that the rate was stable.",
    "  Not estimable            too few deaths, or too few years with deaths,",
    "                           to fit the model at all",
    "",
    "Sheets:",
    "  All-cause            annual percentage change (95% CI) by country and age",
    "                       group; * marks an FDR-adjusted p below 0.05",
    "  Chapters <age>       the same, one sheet per age group, by ICD-10 chapter",
    "  Chapter counts       number of chapters in each class, by country and age,",
    "                       AND the share of deaths those chapters carry. The two",
    "                       can disagree sharply: a country may have a third of",
    "                       its chapters rising while they hold a fraction of its",
    "                       deaths, because the largest chapters - neoplasms and",
    "                       circulatory disease at the older ages - are not among",
    "                       them.",
    "  Death share by class percentage of deaths in each class, at ICD-10 code",
    "                       level, by country and age group (the data behind",
    "                       Figure 3)",
    "",
    "Why 'Chapter counts' and 'Death share by class' do not match. They are",
    "different fits, not two views of one fit. A chapter is one model on pooled",
    "counts; the codes inside it are ~60 separate models on much smaller counts,",
    "so most of them cannot resolve a 1%/year change even when the chapter can.",
    "Comparing the death-weighted columns of the two sheets isolates that loss",
    "of power; comparing counts with shares within 'Chapter counts' isolates the",
    "effect of cause size. Figure 3 of the main text is the code-level sheet.",
    "  Codes most often rising  ICD-10 codes ranked by the number of countries",
    "                       showing a material increase",
    "  Sex all-cause <age>  men, women and the difference between them, by",
    "                       country, one sheet per age group",
    "  Sex chapters <age>   the sex difference by ICD-10 chapter and country",
    "",
    "The complete record, one row per fitted model, is in the accompanying CSV",
    "files listed in Supplementary File 1, Section 8.",
]})

with pd.ExcelWriter(BOOK_DIR / "Supplementary_estimates.xlsx") as xl:
    README.to_excel(xl, sheet_name="Readme", index=False)
    supp_table(all_cause, index="age_group").to_excel(xl, sheet_name="All-cause")
    for age in AGE_ORDER:
        d = chapters[chapters.age_group == age]
        if not d.empty:
            supp_table(d, index="cause_label").to_excel(
                xl, sheet_name=f"Chapters {age}".replace("+", "plus")[:31])
    chapter_class_counts(chapters).to_excel(xl, sheet_name="Chapter counts",
                                            index=False)
    death_share_by_class(codes).to_excel(xl, sheet_name="Death share by class",
                                         index=False)
    codes_most_often_rising(codes).to_excel(
        xl, sheet_name="Codes most often rising", index=False)


def fmt_sex(r):
    def one(prefix):
        a, lo, hi = r[prefix], r[f"{prefix}_lo"], r[f"{prefix}_hi"]
        if not np.isfinite(a) or not np.isfinite(lo):
            return "ne"
        star = "*" if r.get(f"{prefix}_p_bh", 1) < 0.05 else ""
        return f"{a:+.2f} ({lo:+.2f}, {hi:+.2f}){star}"
    return pd.Series({"Men": one("apc_male"), "Women": one("apc_female"),
                      "Difference": one("apc_diff")})


# Sex-specific sheets, appended to the same workbook.
with pd.ExcelWriter(BOOK_DIR / "Supplementary_estimates.xlsx",
                    mode="a", engine="openpyxl") as xl:
    t = sex_all.join(sex_all.apply(fmt_sex, axis=1))
    for age in SEX_AGES:
        d = t[t.age_group == age]
        if d.empty:
            continue
        (d.set_index("country")[["Men", "Women", "Difference", "category_sexdiff",
                                 "apc_diff_mde80", "deaths_male", "deaths_female"]]
         .rename(columns={"category_sexdiff": "Verdict",
                          "apc_diff_mde80": "Min. detectable difference"})
         .round(2)
         .to_excel(xl, sheet_name=f"Sex all-cause {age}".replace("+", "plus")[:31]))
    tc = sex_chapters.join(sex_chapters.apply(fmt_sex, axis=1))
    for age in [a for a in AGE_ORDER if a in set(tc.age_group.dropna())]:
        d = tc[tc.age_group == age]
        (d.pivot_table(index="cause_label", columns="country",
                       values="Difference", aggfunc="first", observed=False)
         .to_excel(xl, sheet_name=f"Sex chapters {age}"[:31]))
print("wrote", BOOK_DIR / "Supplementary_estimates.xlsx")